# Fine-tuning DistilBERT Base Model for Extractive Question Answering with Stanford Question Answering Dataset (SQuAD) 

## 1. Introducción

Este notebook demuestra el fine-tuning de DistilBERT para respuesta extractiva de preguntas usando el Stanford Question Answering Dataset (SQuAD). A diferencia de los enfoques generativos que crean texto, la respuesta extractiva de preguntas identifica y extrae el fragmento exacto de texto de un contexto que responde una pregunta.

Este ejercicio cubre las siguientes tareas principales:
- Preparar el conjunto de datos SQuAD tokenizando contextos y preguntas, mientras se mantiene la alineación entre las posiciones de caracteres y los índices de tokens.
- Fine-tuning DistilBERT para predecir las posiciones de inicio y fin de los fragmentos de respuesta.
- Evaluar el rendimiento usando las métricas de Exact Match y F1 de SQuAD, y luego demostrar el modelo con ejemplos de respuesta de preguntas.

## 2. Setup Inicial

### 1.1 Comprobación de la GPU

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 2.2 Instalación de las librerías

Las siguientes bibliotecas son necesarias para este pipeline:
- transformers: proporciona modelos transformer preentrenados como DistilBERT y tokenizadores.
- datasets: permite la carga y procesamiento eficiente de SQuAD.
- evaluate: contiene las métricas de evaluación que miden la calidad de las predicciones.
- protobuf==3.20.3: asegura la compatibilidad con otras bibliotecas.

In [ ]:
pip install -q transformers datasets evaluate protobuf==3.20.3

## 3. Carga del *Dataset*, Tokenizador y Modelo

### 3.1 SQuAD Dataset

El Stanford Question Answering Dataset (SQuAD) es un punto de referencia para la respuesta extractiva de preguntas, que contiene más de 100,000 pares de preguntas y respuestas de artículos de Wikipedia. Cada ejemplo consiste en un párrafo de contexto, una pregunta y respuestas que son fragmentos exactos de texto del contexto.

La estructura del conjunto de datos incluye tres campos clave: `context`, `question` y `answers`; estás últimas son un diccionario con `text` y `answer_start` (el índice de carácter donde comienza la respuesta). Esta posición de carácter es crucial para convertir a posiciones de token durante el entrenamiento.

Los ejemplos de entrenamiento contienen exactamente una respuesta por pregunta, mientras que los ejemplos de validación pueden tener múltiples respuestas aceptables de diferentes anotadores. Durante la evaluación, las predicciones se comparan contra todas las respuestas válidas y se toma la mejor puntuación.

In [ ]:
from datasets import load_dataset

# Load SQuAD dataset: contains context, question, and answer (with character position)
# Returns DatasetDict with 'train' and 'validation' splits
squad = load_dataset("squad")

### 3.2 Declaración del tokenizador

Los modelos transformer requieren que el texto se convierta en tokens numéricos. Para la respuesta de preguntas, el tokenizador combina la pregunta y el contexto en una sola secuencia usando tokens especiales: [CLS] al inicio, [SEP] separando la pregunta del contexto, y [SEP] al final. Esta estructura permite a DistilBERT distinguir la pregunta del contexto.

El tokenizador distilbert-base-cased es sensible a mayúsculas y minúsculas. El tokenizador también proporciona mapeos de desplazamiento que vinculan cada token con su posición en el texto original, lo que permite convertir las posiciones de las respuestas de caracteres a tokens para el entrenamiento.

In [ ]:
from transformers import AutoTokenizer

# load distilBERT tokenizer
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

### 3.3 Inicialización del Modelo

DistilBERT se inicializa con una cabeza para responder a preguntas. La advertencia sobre los pesos `qa_outputs` recién inicializados se espera, ya que estas capas específicas de la tarea se inicializan aleatoriamente mientras que las capas preentrenadas conservan su comprensión del lenguaje. El ajuste fino entrena ambos componentes para la extracción de fragmentos.

In [ ]:
from transformers import AutoModelForQuestionAnswering

# initialize distilBERT with qa head
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

## 4. Preprocessing

### 4.1 Preprocesamiento del Conjunto de Datos de Entrenamiento

El preprocesamiento maneja contextos largos y convierte posiciones de respuesta basadas en caracteres a índices de tokens. Muchos contextos de SQuAD exceden las embeddings de posición máximos de DistilBERT, por lo que el código un desplazamiento de 128 tokens, dividiendo contextos largos en fragmentos superpuestos.

La tokenización mantiene las preguntas intactas mientras divide los contextos, crea múltiples características a partir de ejemplos largos y mapea de carácter a token. Para cada característica, los mapeos de desplazamiento determinan si la respuesta cae dentro de ese fragmento de contexto.

El proceso de etiquetado identifica tokens de contexto, asigna etiquetas (0, 0) cuando las respuestas caen fuera de los fragmentos, y de lo contrario encuentra el primer y último token del fragmento de respuesta para crear las etiquetas `start_positions` y `end_positions`. Los espacios en blanco adicionales se eliminan de las preguntas para evitar desperdiciar tokens.

In [ ]:
max_length = 512 # maximum sequence length
stride = 128 # sliding window stride for handling long contexts

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]] # strip extra whitespace from questions
    
    # tokenize question-context pairs
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second", # keep questions intact, split contexts
        stride=stride,
        return_overflowing_tokens=True, # create multiple features from long contexts
        return_offsets_mapping=True, # get character-to-token alignment
        padding="max_length",
    )

    # extract offset mappings and sample mappings for label generation
    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    # generate start and end position labels for each feature
    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # if answer is not fully inside the context chunk, label as (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # find token indices for answer start and end using offset mappings
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

Este preprocesamiento se aplica usando `Dataset.map()` con `batched=True` para eficiencia y `remove_columns=squad["train"].column_names` para reemplazar la estructura original con entradas tokenizadas y etiquetas de posición. El argumento `batched=True` es esencial para la eficiencia. 

In [ ]:
# Apply preprocessing to training set
train_dataset = squad["train"].map(
    preprocess_training_examples,
    batched=True, # process multiple examples at once for efficiency
    remove_columns=squad["train"].column_names, # replace original structure with tokenized inputs and labels
)

len(squad["train"]), len(train_dataset)

### 4.2 Preprocesamiento del Conjunto de Datos de Validación

El preprocesamiento de validación no requiere etiquetas de respuesta ya que el modelo las predice. Sin embargo, se debe preservar información para mapear las predicciones de vuelta a los ejemplos originales y extraer el texto de la respuesta.

Aunque los modelos producen predicciones a nivel de token,  se necesitan fragmentos de texto. Múltiples características de contextos divididos producen diferentes predicciones que deben ser agregadas.

Los mapeos de desplazamiento se mantienen, pero los desplazamientos de los tokens de pregunta se establecen en None, asegurando que solo se consideren los tokens de contexto al extraer fragmentos de respuesta. Los IDs de ejemplo se preservan porque las ventanas deslizantes crean múltiples características por cada ejemplo, y `overflow_to_sample_mapping` rastrea de qué ejemplo original provino cada característica.

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]  # strip extra whitespace from questions

    # same as training, but no labels needed
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx]) # store example ID to group features from same example during evaluation

        # set question token offsets to None, keeping only context token offsets
        # this helps identify valid answer positions during post-processing
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

Este preprocesamiento se aplica al conjunto de validación, reemplazando las columnas originales con características procesadas que contienen entradas tokenizadas, mapeos de desplazamiento e IDs de ejemplo.

In [ ]:
# apply preprocessing to validation set
validation_dataset = squad["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=squad["validation"].column_names,
)

len(squad["validation"]), len(validation_dataset)

## 5. Fine-tuning el modelo con Trainer API

### 5.1 Definición de Argumentos

`TrainingArguments` encapsula los hiperparámetros que controlan el proceso de entrenamiento

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-squad", # output directory for checkpoints
    eval_strategy="epoch", # evaluate at end of each epoch
    save_strategy="epoch", # save checkpoint at end of each epoch
    learning_rate=2e-5, # standard learning rate for BERT fine-tuning
    num_train_epochs=1, # one pass through dataset (increase for better performance)
    weight_decay=0.01, # regularization to prevent overfitting
    fp16=True, # mixed precision training for speed
    report_to="none", # disable logging
)

### 5.2 Creación del Entrenador

Trainer maneja el bucle de entrenamiento, la configuración del optimizador y la evaluación. Cada paso procesa un *batch*: *forward pass*, cálculo de pérdida, retropropagación y actualizaciones de pesos.

In [ ]:
from transformers import Trainer

# trainer orchestrates the training loop, gradient computation, and optimization
trainer = Trainer(
    model=model, # distilBERT with qa head
    args=args, # training configuration
    train_dataset=train_dataset, # preprocessed training features with labels
    eval_dataset=validation_dataset, # preprocessed validation features
    processing_class=tokenizer, # tokenizer for batch collation
)

### 5.3 Entrenamiento del Modelo

El entrenamiento se inicia con `trainer.train()`, procesando 88,729 características a lo largo de 11,092 pasos en una época. El entrenamiento tomó ~30 minutos a ~49 ejemplos/segundo.

In [ ]:
trainer.train()

## 6. Evaluación

La evaluación convierte las predicciones del modelo a fragmentos de texto y maneja múltiples características por ejemplo. La estrategia considera las 20 primeras posiciones de inicio y las 20 primeras posiciones de fin, evalúa combinaciones válidas y puntúa cada una sumando los logits. Las respuestas inválidas se filtran: aquellas fuera del contexto (mediante desplazamientos `None`), respuestas de longitud negativa y respuestas que exceden 30 tokens.

Para contextos divididos, se selecciona la respuesta válida con mayor puntuación entre todas las características, o una cadena vacía si no existe ninguna. **SQuAD utiliza Coincidencia Exacta (EM), que otorga una puntuación de 1 para coincidencias exactas, y Puntuación F1, que trata la predicción y la verdad fundamental como bolsas de tokens y calcula la media armónica de precisión-recuperación.**
```

In [ ]:
from tqdm.auto import tqdm
import collections
import numpy as np
import evaluate

n_best = 20  # consider top 20 start/end positions
max_answer_length = 30  # maximum answer span length in tokens
metric = evaluate.load("squad") # load SQuAD metric for evaluation (exact match and f1)

def compute_metrics(start_logits, end_logits, features, examples):
    # map each example to its features
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    
    # process each example to find best answer across all its features
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # collect candidate answers from all features for this example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            # get top 20 start and end positions by logit score
            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            
            # Eealuate all valid (start, end) combinations
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # skip if answer is not fully in context (offset is None for question tokens)
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # skip invalid answers: negative length or too long
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    # extract answer text from context using offsets
                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # select answer with highest logit score (or empty string if no valid answers)
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    # format ground truth answers and compute SQuAD metrics (em and f1)
    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

Después del entrenamiento, `trainer.predict()` ejecuta la inferencia en el conjunto de validación para obtener los logits de inicio/fin. Estos logits se pasan a `compute_metrics()` con las características y ejemplos de validación para el posprocesamiento, identificación de fragmentos y evaluación contra la verdad fundamental.

In [ ]:
# run inference on validation set and get start/end logits
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions

# post-process predictions to extract answer spans and compute metrics
compute_metrics(start_logits, end_logits, validation_dataset, squad["validation"])

## 7. Performance Comparison

El modelo alcanza 73.82% de EM y 82.74% de F1 en la validación de SQuAD. Para demostrar la mejora, el DistilBERT base se compara con la versión ajustada finamente usando pipelines de Hugging Face.

In [ ]:
from transformers import pipeline

# create qa pipeline for base model (no fine-tuning)
base_qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-cased"
)

# create qa pipeline for fine-tuned model
tuned_qa_pipeline = pipeline(
    task="question-answering",
    model=trainer.model,
    tokenizer=tokenizer
)

# test cases
test_cases = [
    {
        "context": """
        Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, is designed to provide Large Language Models (LLMs) with additional information from an external knowledge source.
        The main purpose of RAG is to generate more accurate and contextual answers while reducing hallucinations.
        In RAG, factual knowledge is separated from the LLMs reasoning capability and stored in an external knowledge source.
        This technique is analogous to taking an open-book exam: "RAG is to LLMs what an open-book exam is to humans".
        """,
        "question": "What is the primary function of Retrieval-Augmented Generation (RAG) in LLM applications?"
    },
    {
        "context": """
        GPT-3 is an autoregressive language model released by OpenAI in 2020.
        This model is characterized by its scale, which includes 96 Transformer decoder layers.
        The model possesses 175 billion parameters and was trained using 300 billion tokens from several sources.
        Furthermore, GPT-3 has a 2048-token-long context window, which represents the amount of text the model can consider when generating a response.
        """,
        "question": "How many parameters does the GPT-3 model have?"
    }, 
    {
        "context": """
        The transformer architecture was introduced by Vaswani et al. in the paper "Attention Is All You Need" (2017).
        The architecture eliminates Recurrent Neural Networks (RNNs) and uses only attention mechanisms.
        While the quality of MT was improved, the architecture especially drastically improves the training speed because it deals with sequential data in parallel, allowing the use of GPUs/TPUs.
        Language Models like Gemini and ChatGPT are based on transformers.
        """,
        "question": "Which research paper introduced the Transformer architecture and in what year?"
    },
    {
        "context": """
        Long short-term memory (LSTM) models were developed to address long-term dependencies, solving issues found in traditional Recurrent Neural Networks (RNNs) that often exhibit short-term memory.
        LSTMs introduce the cell state, which is key to their functionality.
        The cell state acts as the long-term memory of the LSTM, designed to carry information across many time steps while undergoing minimal modifications.
        In contrast, the hidden state represents the short-term, immediate output, used for predictions and communication with other layers.
        """,
        "question": "What role does the cell state play within the Long short-term memory (LSTM) architecture?"
    },
    {
        "context": """
        Word embeddings are designed not only to reflect similarity but also to preserve semantic and syntactic relationships.
        These relationships are observed in the difference vectors between words.
        This property allows for analogies or queries to be expressed as equations
        Furthermore, these sophisticated relationships can be leveraged in other applications, such as assisting with machine translation.
        """,
        "question": "How do word embeddings encode semantic and syntactic relationships between words?"
    }
]

# compare base and fine-tuned model on each test case
for i, test_case in enumerate(test_cases, 1):
    context = test_case["context"].strip()
    question = test_case["question"]

    base_result = base_qa_pipeline(question=question, context=context)
    base_answer = base_result["answer"]
    base_score = base_result["score"]

    tuned_result = tuned_qa_pipeline(question=question, context=context)
    tuned_answer = tuned_result["answer"]
    tuned_score = tuned_result["score"]

    print(f"Test case #{i}\n")
    print(f"Context: {context}\n")
    print(f"Question: {question}\n")

    print(f"Base model:\n")
    print(f"- Answer: {base_answer}\n")
    print(f"- Confidence: {base_score:.4f}\n")

    print(f"Fine-tuned model:\n")
    print(f"- Answer: {tuned_answer}\n")
    print(f"- Confidence: {tuned_score:.4f}\n")

## 8. Conclusion

El pipeline completo para el ajuste fino de un transformer para respuesta extractiva de preguntas incluye:

- **Preparación de Datos**: Carga de SQuAD, preprocesamiento con ventanas deslizantes, mantenimiento de alineación de carácter a token y generación de etiquetas a nivel de token.
- **Entrenamiento del Modelo**: Ajuste fino de DistilBERT para predecir fragmentos de respuesta sobre 88,729 características en ~30 minutos.
- **Evaluación**: Posprocesamiento de predicciones, agregación de características, cálculo de métricas de SQuAD (73.82% EM, 82.74% F1).
- **Análisis de Rendimiento**: Demostración de mejora sobre el modelo base en pruebas específicas del dominio.